In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd
from pprint import pprint
from IPython.display import Image, Markdown, display

_here = Path.cwd().resolve()
_repo_root = next(path for path in (_here, *_here.parents) if (path / "configs" / "cfg.py").exists())
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from data.share_data import load_share_data
from scripts.mpc import plot_mpc_outputs, run_admm_mpc_ev, run_local_mpc_ev
from utils.records import compute_ev_agent_cost_summary, compute_ev_cost_summary
from utils.run_artifacts import load_experiment_context


In [ ]:
run_dir = Path("artifacts/runs/20260510_172102_43adfd99")
cfg, run_dir = load_experiment_context(run_dir)

cfg.env.ev_enabled = True
cfg.model.action_dim = 3
cfg.env.ev_departure_constraint_mode = "hard"
cfg.env.ev_hard_projection_enabled = True
cfg.env.ev_emergency_charging_enabled = False
cfg.env.ev_emergency_window_hours = float(cfg.env.ev_emergency_window_hours)
cfg.env.ev_emergency_strategy = "required_power"

cfg.env.ev_capacity_kwh = (60.0, 60.0, 60.0)
cfg.env.ev_soc_min = 0.10
cfg.env.ev_soc_max = 0.95
cfg.env.ev_arrival_soc = 0.15
cfg.env.ev_departure_soc_req = 0.90
cfg.env.ev_max_charge_kw = (11.0, 11.0, 11.0)
cfg.env.ev_efficiency = 0.95
cfg.env.ev_arrival_step = 72
cfg.env.ev_departure_step = 28
cfg.reward.ev_departure_penalty_weight = 0.0
cfg.reward.ev_soc_regularization_weight = 0.005
cfg.reward.ev_projection_penalty_weight = 0.0
cfg.reward.ev_emergency_penalty_weight = 0.0

share_data = load_share_data(run_dir / "share_data", cfg)
display(Markdown("# MPC + EV Hard Constraint"))
display(pd.DataFrame([{
    "run_dir": str(run_dir),
    "eval_start": cfg.data.eval_start_date,
    "eval_end": cfg.data.eval_end_date,
    "eval_episodes": int(share_data.eval["price"].shape[0]),
    "ev_enabled": bool(cfg.env.ev_enabled),
    "action_dim": int(cfg.model.action_dim),
    "ev_constraint_mode": cfg.env.ev_departure_constraint_mode,
    "ev_capacity_kwh": cfg.env.ev_capacity_kwh,
    "ev_arrival_soc": float(cfg.env.ev_arrival_soc),
    "ev_departure_soc_req": float(cfg.env.ev_departure_soc_req),
    "ev_max_charge_kw": cfg.env.ev_max_charge_kw,
    "ev_arrival_step": int(cfg.env.ev_arrival_step),
    "ev_departure_step": int(cfg.env.ev_departure_step),
    "ev_departure_penalty_weight": float(cfg.reward.ev_departure_penalty_weight),
    "ev_hard_projection_enabled": bool(cfg.env.ev_hard_projection_enabled),
    "ev_emergency_charging_enabled": bool(cfg.env.ev_emergency_charging_enabled),
    "ev_emergency_window_hours": float(cfg.env.ev_emergency_window_hours),
}]))


In [ ]:
local_records = run_local_mpc_ev(cfg, run_dir, share_data, ev_mode=cfg.env.ev_departure_constraint_mode)
admm_record = run_admm_mpc_ev(cfg, run_dir, share_data, ev_mode=cfg.env.ev_departure_constraint_mode)
records = {"local": local_records, "admm": admm_record}
metrics_df = pd.concat([local_records["perfect"]["metrics_df"], local_records["lstm"]["metrics_df"], admm_record["metrics_df"]], ignore_index=True)
display(Markdown("## Metrics"))
display(metrics_df)
pprint({"record_root": str(run_dir / "results")})


In [ ]:
display(Markdown("## EV and total cost summary"))
rollouts = {
    "local_perfect": local_records["perfect"]["rollout"],
    "local_lstm": local_records["lstm"]["rollout"],
    "admm_lstm": admm_record["rollout"],
}
summary_rows = []
agent_rows = []
for key, rollout in rollouts.items():
    _, summary = compute_ev_cost_summary(rollout, cfg)
    summary.insert(0, "run_key", key)
    summary.insert(1, "controller", rollout.meta.get("controller", key))
    summary_rows.append(summary)
    agent_summary = compute_ev_agent_cost_summary(rollout, cfg)
    agent_summary.insert(0, "run_key", key)
    agent_summary.insert(1, "controller", rollout.meta.get("controller", key))
    agent_rows.append(agent_summary)

display(Markdown("### Total cost"))
display(pd.concat(summary_rows, ignore_index=True))
display(Markdown("### Per-agent cost"))
display(pd.concat(agent_rows, ignore_index=True))

def ev_departure_summary(rollout):
    agent = rollout.agent_df.copy()
    required_soc = float(rollout.meta.get("ev_departure_soc_req", cfg.env.ev_departure_soc_req))
    departure_step = int(rollout.meta.get("ev_departure_step", cfg.env.ev_departure_step))
    dt_hours = float(rollout.meta.get("dt_hours", cfg.env.dt_hours))
    departures = agent.loc[agent["step"].astype(int) == departure_step]
    if departures.empty:
        departures = agent.sort_values(["episode_idx", "agent_id", "step"]).groupby(["episode_idx", "agent_id"], as_index=False).tail(1)
    if "ev_departure_gap" not in departures.columns:
        departures = departures.copy()
        departures["ev_departure_gap"] = (required_soc - departures["ev_soc"].astype(float)).clip(lower=0.0)
    energy = agent.assign(ev_energy_kwh=agent["ev_charge_kw"].astype(float).clip(lower=0.0) * dt_hours).groupby("agent_id", as_index=False)["ev_energy_kwh"].sum()
    cost_col = "ev_charging_cost_eur" if "ev_charging_cost_eur" in agent.columns else None
    cost = agent.groupby("agent_id", as_index=False)[cost_col].sum().rename(columns={cost_col: "ev_cost_eur"}) if cost_col else pd.DataFrame({"agent_id": sorted(agent["agent_id"].unique()), "ev_cost_eur": 0.0})
    departure = departures.groupby("agent_id", as_index=False).agg(departure_soc=("ev_soc", "min"), departure_gap=("ev_departure_gap", "max"))
    result = departure.merge(energy, on="agent_id", how="outer").merge(cost, on="agent_id", how="outer")
    result = result.rename(columns={"agent_id": "agent"}).sort_values("agent")
    result["required_soc"] = required_soc
    result["satisfied"] = result["departure_gap"].fillna(0.0) <= 1e-6
    return result.loc[:, ["agent", "departure_soc", "required_soc", "departure_gap", "satisfied", "ev_energy_kwh", "ev_cost_eur"]]

for key, rollout in rollouts.items():
    display(Markdown(f"### Departure check: {key}"))
    display(ev_departure_summary(rollout))


In [ ]:
figures = plot_mpc_outputs(records, run_dir)
_plot_specs = [
    ("1. Power balance", "mpc_power_balance"),
    ("2. Price", "mpc_price"),
    ("3. Battery power and SoC", "mpc_battery_soc"),
    ("4. Voltage", "mpc_voltage"),
    ("5. Net load", "mpc_net_load"),
    ("6. EV SOC", "mpc_ev_soc"),
    ("7. EV charging power", "mpc_ev_charge_kw"),
    ("8. EV departure SOC requirement check", "mpc_ev_departure_check"),
    ("9. Total cost", "mpc_total_cost"),
    ("10. Cost components", "mpc_cost_components"),
    ("11. Cumulative cost components", "mpc_cumulative_cost_components"),
]
for title, name in _plot_specs:
    path = run_dir / "figures" / f"{name}.png"
    if path.exists():
        print(f"{title} - {cfg.env.ev_departure_constraint_mode}")
        display(Image(filename=str(path)))
        if name in figures:
            plt.close(figures[name])
    else:
        print(f"[missing] {title}: {path}")
